In [1]:
!pip install streamlit openai numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 20.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 49.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [3]:
%%writefile seed.py

import sqlite3

jobs = [
    ("Python Backend Developer", "TechNova", "Entry-level Python, Flask, REST APIs, SQL"),
    ("Junior Software Engineer", "CloudWorks", "Python, Django, backend development, databases"),
    ("Data Analyst", "Insight Labs", "Python, pandas, SQL, data visualization"),
    ("Machine Learning Intern", "AI Solutions", "Python, machine learning, scikit-learn"),
    ("Frontend Developer", "WebSpark", "JavaScript, React, HTML, CSS"),
    ("Backend Engineer", "DevStack", "Python, FastAPI, APIs, PostgreSQL"),
    ("Software Developer Intern", "CodeCraft", "Python, Git, problem solving"),
    ("Data Science Intern", "Analytics Hub", "Python, statistics, machine learning"),
    ("Java Developer", "EnterpriseTech", "Java, Spring Boot, SQL"),
    ("Full Stack Developer", "StartupX", "Python, React, APIs, web development"),
    ("Cloud Engineer Intern", "CloudNine", "AWS, Python, Linux, Docker"),
    ("Automation Engineer", "AutoTech", "Python scripting, automation, testing"),
    ("Junior QA Engineer", "QualityFirst", "Python, software testing, Selenium"),
    ("Database Developer", "DataCore", "SQL, PostgreSQL, database design"),
    ("AI Research Intern", "FutureAI", "Python, deep learning, NLP"),
]

conn = sqlite3.connect("jobs.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS jobs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    company TEXT,
    description TEXT
)
""")

cursor.execute("DELETE FROM jobs")

cursor.executemany(
    "INSERT INTO jobs (title, company, description) VALUES (?, ?, ?)",
    jobs
)

conn.commit()
conn.close()

print("Database created successfully!")
print(f"{len(jobs)} job listings added.")

Overwriting seed.py


In [4]:
import sqlite3

jobs = [
    ("Python Backend Developer", "TechNova", "Entry-level Python, Flask, REST APIs, SQL"),
    ("Junior Software Engineer", "CloudWorks", "Python, Django, backend development, databases"),
    ("Data Analyst", "Insight Labs", "Python, pandas, SQL, data visualization"),
    ("Machine Learning Intern", "AI Solutions", "Python, machine learning, scikit-learn"),
    ("Frontend Developer", "WebSpark", "JavaScript, React, HTML, CSS"),
    ("Backend Engineer", "DevStack", "Python, FastAPI, APIs, PostgreSQL"),
    ("Software Developer Intern", "CodeCraft", "Python, Git, problem solving"),
    ("Data Science Intern", "Analytics Hub", "Python, statistics, machine learning"),
    ("Java Developer", "EnterpriseTech", "Java, Spring Boot, SQL"),
    ("Full Stack Developer", "StartupX", "Python, React, APIs, web development"),
    ("Cloud Engineer Intern", "CloudNine", "AWS, Python, Linux, Docker"),
    ("Automation Engineer", "AutoTech", "Python scripting, automation, testing"),
    ("Junior QA Engineer", "QualityFirst", "Python, software testing, Selenium"),
    ("Database Developer", "DataCore", "SQL, PostgreSQL, database design"),
    ("AI Research Intern", "FutureAI", "Python, deep learning, NLP"),
]

conn = sqlite3.connect("jobs.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS jobs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    company TEXT,
    description TEXT
)
""")

cursor.execute("DELETE FROM jobs")

cursor.executemany(
    "INSERT INTO jobs (title, company, description) VALUES (?, ?, ?)",
    jobs
)

conn.commit()

print("Database created!")
print("Jobs added:", cursor.execute("SELECT COUNT(*) FROM jobs").fetchone()[0])

conn.close()

Database created!
Jobs added: 15


In [5]:
!pip install sentence-transformers

In [6]:
from sentence_transformers import SentenceTransformer

print("Embedding library is working!")

Embedding library is working!


In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [13]:
import sqlite3
import json

conn = sqlite3.connect("jobs.db")
cursor = conn.cursor()

# Add a column to store vectors
try:
  cursor.execute("ALTER TABLE jobs ADD COLUMN embedding TEXT")
except sqlite3.OperationalError:
  pass

# Get all jobs
jobs = cursor.execute(
"SELECT id, title, company, description FROM jobs"
).fetchall()

# Generate vectors
for job in jobs:
  job_id, title, company, description = job

text = f"{title} {company} {description}"

vector = model.encode(text).tolist()

cursor.execute(
"UPDATE jobs SET embedding = ? WHERE id = ?",
(json.dumps(vector), job_id)
)

conn.commit()

print("Embeddings saved for", len(jobs), "jobs!")
conn.close()

Embeddings saved for 15 jobs!


In [23]:
import sqlite3
import json
import numpy as np

def search_jobs(query, top_k=5):
    # Convert search query into vector
    query_vector = model.encode(query)

    conn = sqlite3.connect("jobs.db")
    cursor = conn.cursor()

    jobs = cursor.execute(
        "SELECT id, title, company, description, embedding FROM jobs"
    ).fetchall()

    results = []

    for job in jobs:
        job_id, title, company, description, embedding_json = job

        job_vector = np.array(json.loads(embedding_json))

        # Calculate cosine similarity
        similarity = np.dot(query_vector, job_vector) / (
            np.linalg.norm(query_vector) * np.linalg.norm(job_vector)
        )

        results.append({
            "title": title,
            "company": company,
            "description": description,
            "score": float(similarity)
        })

    conn.close()

    # Rank from most similar to least similar
    results.sort(key=lambda x: x["score"], reverse=True)

    return results[:top_k]

In [24]:
print("Testing semantic search...")

results = search_jobs("Python backend developer", top_k=5)

print("Number of results:", len(results))

for job in results:
    print(job["title"], "-", job["company"], "-", round(job["score"], 3))

Testing semantic search...


TypeError: the JSON object must be str, bytes or bytearray, not NoneType

In [25]:
import sqlite3
import json

conn = sqlite3.connect("jobs.db")
cursor = conn.cursor()

jobs = cursor.execute(
    "SELECT id, title, company, description FROM jobs"
).fetchall()

for job in jobs:
    job_id, title, company, description = job

    text = f"{title} {company} {description}"
    vector = model.encode(text).tolist()

    cursor.execute(
        "UPDATE jobs SET embedding = ? WHERE id = ?",
        (json.dumps(vector), job_id)
    )

conn.commit()

# Verify
missing = cursor.execute(
    "SELECT COUNT(*) FROM jobs WHERE embedding IS NULL"
).fetchone()[0]

print("Missing embeddings:", missing)
print("Embeddings repaired!")
conn.close()

Missing embeddings: 0
Embeddings repaired!


In [27]:
import numpy as np
import sqlite3
import json

def search_jobs(query, top_k=5):

    query_vector = model.encode(query)

    conn = sqlite3.connect("jobs.db")

    jobs = conn.execute(
        "SELECT title, company, description, embedding FROM jobs"
    ).fetchall()

    conn.close()

    results = []

    for title, company, description, embedding_json in jobs:

        if embedding_json is None:
            continue

        job_vector = np.array(json.loads(embedding_json))

        score = np.dot(query_vector, job_vector) / (
            np.linalg.norm(query_vector) * np.linalg.norm(job_vector)
        )

        results.append({
            "title": title,
            "company": company,
            "description": description,
            "score": float(score)
        })

    results.sort(key=lambda x: x["score"], reverse=True)

    return results[:top_k]

In [28]:
results = search_jobs("entry level Python backend developer")

print("Number of results:", len(results))

for job in results:
    print(f"\n{job['title']} — {job['company']}")
    print("Score:", round(job["score"], 3))
    print("Description:", job["description"])

Number of results: 5

Python Backend Developer — TechNova
Score: 0.769
Description: Entry-level Python, Flask, REST APIs, SQL

Junior Software Engineer — CloudWorks
Score: 0.662
Description: Python, Django, backend development, databases

Full Stack Developer — StartupX
Score: 0.634
Description: Python, React, APIs, web development

Backend Engineer — DevStack
Score: 0.612
Description: Python, FastAPI, APIs, PostgreSQL

Software Developer Intern — CodeCraft
Score: 0.596
Description: Python, Git, problem solving


In [29]:
!pip install groq

In [30]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

Paste your Groq API key: ··········


In [31]:
from groq import Groq
import json

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Define the search tool available to the AI
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_jobs",
            "description": "Search the job database for jobs relevant to the user's query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Description of the type of job the user wants"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

def ask_job_assistant(user_query):

    messages = [
        {
            "role": "system",
            "content": """You are an intelligent job search assistant.
            Search the job database using the search_jobs tool.
            Recommend the most relevant jobs and explain briefly why
            each matches the user's request.
            Only recommend jobs returned by the search tool.
            Be concise and helpful."""
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    # Ask the LLM what it wants to do
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # If the AI decides to use a tool
    if message.tool_calls:

        messages.append(message)

        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if tool_name == "search_jobs":

                query = arguments["query"]

                results = search_jobs(query, top_k=5)

                # Send search results back to the AI
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_name,
                    "content": json.dumps(results)
                })

        # AI now explains the retrieved results
        final_response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages
        )

        return final_response.choices[0].message.content

    else:
        return message.content

In [33]:
answer = ask_job_assistant(
    "I am a beginner with Python and SQL. Find me entry-level backend jobs."
)

print(answer)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [34]:
def ask_job_assistant(user_query):

    messages = [
        {
            "role": "system",
            "content": """You are an intelligent job search assistant.
            Use the search_jobs tool to search the job database.
            Recommend relevant jobs and briefly explain why they match.
            Only recommend jobs returned by the tool.
            Be concise and helpful."""
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    if message.tool_calls:

        messages.append(message)

        for tool_call in message.tool_calls:

            arguments = json.loads(tool_call.function.arguments)

            results = search_jobs(arguments["query"], top_k=5)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "search_jobs",
                "content": json.dumps(results)
            })

        final_response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages
        )

        return final_response.choices[0].message.content

    return message.content

In [35]:
answer = ask_job_assistant(
    "I am a beginner with Python and SQL. Find me entry-level backend jobs."
)

print(answer)

**Entry‑level backend roles that fit your Python/SQL skills**

| Job | Company | Why it matches |
|-----|---------|----------------|
| Junior Software Engineer | CloudWorks | Explicitly “backend development” with Python and Django, plus database work—perfect for a beginner. |
| Python Backend Developer | TechNova | Labelled “entry‑level,” focuses on Flask, REST APIs and SQL, matching your exact skill set. |
| Backend Engineer | DevStack | Uses Python and FastAPI with PostgreSQL; while not tagged “entry‑level,” it’s a light‑weight backend role ideal for someone just starting. |

Feel free to apply or let me know if you’d like more details on any of these!


In [36]:
import ipywidgets as widgets
from IPython.display import display, Markdown

search_box = widgets.Text(
    placeholder="e.g. entry-level Python backend jobs",
    description="Job search:",
    layout=widgets.Layout(width="80%")
)

search_button = widgets.Button(
    description="Find Jobs",
    button_style="primary"
)

output = widgets.Output()

def on_search_clicked(button):
    with output:
        output.clear_output()

        query = search_box.value

        if not query.strip():
            print("Please enter a search query.")
            return

        print("Searching for relevant jobs...\n")

        try:
            response = ask_job_assistant(query)
            display(Markdown(response))
        except Exception as e:
            print("Error:", e)

search_button.on_click(on_search_clicked)

display(search_box, search_button, output)

Text(value='', description='Job search:', layout=Layout(width='80%'), placeholder='e.g. entry-level Python bac…

Button(button_style='primary', description='Find Jobs', style=ButtonStyle())

Output()